In [18]:
# main.py
# Member 3 : ties every module together into one interactive text dashboard.

import os
import auth
import events
import certificate
import data_store

def get_valid_choice(prompt, valid_options):
       while True:
        choice = input(prompt).strip()
           
        if choice in valid_options:
            return choice
        print("Invalid choice, try again.")

def organizer_menu(username):
    while True:
        print("\n===== ORGANIZER MENU =====")
        print("1. Post Event")
        print("2. Review Applicants")
        print("3. Complete Event")
        print("4. Logout")

        choice = get_valid_choice("Enter choice: ", ["1", "2", "3", "4"])

        if choice == "1":

            print("\n===== POST EVENT =====")

            event_id = input("Enter event ID: ").strip()
            name = input("Enter event name: ").strip()
            description = input("Enter event description: ").strip()

            required_skills = {}

            print("\nEnter required skills.")
            print("Example: Python:3, CAD:2")
            print("Enter 'none' if no specific skills are required.")

            skills_input = input("Required skills: ").strip()

            if skills_input.lower() != "none" and skills_input != "":

                try:
                    skill_list = skills_input.split(",")

                    for skill_item in skill_list:

                        skill, rating = skill_item.split(":")

                        skill = skill.strip()
                        rating = int(rating.strip())

                        if rating < 0:
                            raise ValueError

                        required_skills[skill] = rating

                except ValueError:
                    print("Invalid skill format.")
                    print("Please use: Python:3, CAD:2")
                    continue

            # Number of volunteer slots
            try:

                slots_needed = int(input("Number of volunteer slots: ").strip())

                if slots_needed <= 0:
                    print("Number of slots must be greater than 0.")
                    continue

            except ValueError:
                print("Please enter a valid whole number.")
                continue

            result = events.create_event(
                event_id,
                name,
                description,
                username,
                required_skills,
                slots_needed
            )

            print(result[1])

        elif choice == "2":

            print("\n===== REVIEW APPLICANTS =====")

            all_events = data_store.load(
                events.EVENTS_FILE
            )

            all_applications = data_store.load(
                events.APPLICATIONS_FILE
            )

            # Find events belonging to this organizer
            organizer_events = []

            for event_id, event in all_events.items():

                if event.get("organizer") == username:
                    organizer_events.append(
                        (event_id, event)
                    )

            if not organizer_events:
                print("You have not posted any events.")
                continue

            print("\nYour events:")

            for event_id, event in organizer_events:

                print(
                    event_id,
                    "-",
                    event.get("name", "Unnamed Event"),
                    "[",
                    event.get("status", "unknown"),
                    "]"
                )

            event_id = input(
                "\nEnter event ID to review: "
            ).strip()

            if event_id not in all_events:
                print("Event does not exist.")
                continue

            # Make sure organizer owns this event
            if all_events[event_id].get("organizer") != username:
                print(
                    "You can only review applicants "
                    "for your own events."
                )
                continue

            event_applications = all_applications.get(
                event_id,
                {}
            )

            if not event_applications:
                print("No applicants for this event.")
                continue

            print("\n===== APPLICANTS =====")

            for applicant, application in event_applications.items():

                print("\nUsername:", applicant)
                print(
                    "Status:",
                    application.get("status")
                )
                print(
                    "Role:",
                    application.get("role")
                )

            applicant = input(
                "\nEnter applicant username to process: "
            ).strip()

            if applicant not in event_applications:
                print("Applicant not found.")
                continue

            # Only pending applications should be processed
            if event_applications[applicant].get("status") != "pending":
                print(
                    "This application has already been processed."
                )
                continue

            decision = get_valid_choice(
                "Accept or reject? (a/r): ",
                ["a", "r"]
            )

            if decision == "a":

                role = input(
                    "Enter role for this volunteer: "
                ).strip()

                if role == "":
                    role = "Volunteer"

                result = events.update_application_status(
                    event_id,
                    applicant,
                    "accepted",
                    role
                )

            else:

                result = events.update_application_status(
                    event_id,
                    applicant,
                    "rejected"
                )

            print(result[1])


        elif choice == "3":

            print("\n===== COMPLETE EVENT =====")

            all_events = data_store.load(
                events.EVENTS_FILE
            )

            all_applications = data_store.load(
                events.APPLICATIONS_FILE
            )

            # Find organizer's events
            organizer_events = []

            for event_id, event in all_events.items():

                if event.get("organizer") == username:
                    organizer_events.append(
                        (event_id, event)
                    )

            if not organizer_events:
                print("You have not posted any events.")
                continue

            print("\nYour events:")

            for event_id, event in organizer_events:

                print(
                    event_id,
                    "-",
                    event.get("name", "Unnamed Event"),
                    "[",
                    event.get("status", "unknown"),
                    "]"
                )

            event_id = input(
                "\nEnter event ID to complete: "
            ).strip()

            if event_id not in all_events:
                print("Event does not exist.")
                continue

            # Organizer can only complete their own event
            if all_events[event_id].get("organizer") != username:
                print(
                    "You can only complete your own events."
                )
                continue

            if all_events[event_id].get("status") == "completed":
                print("This event is already completed.")
                continue

            event_applications = all_applications.get(
                event_id,
                {}
            )

            accepted_students = []

            for student, application in event_applications.items():

                if application.get("status") == "accepted":
                    accepted_students.append(student)

            if not accepted_students:
                print(
                    "There are no accepted volunteers "
                    "for this event."
                )
                continue

            contribution_updates = {}

            print(
                "\nEnter final contribution details "
                "for each accepted volunteer."
            )

            for student in accepted_students:

                print("\nVolunteer:", student)

                try:

                    hours = float(
                        input("Hours contributed: ").strip()
                    )

                    if hours < 0:
                        print(
                            "Hours cannot be negative."
                        )
                        contribution_updates = None
                        break

                except ValueError:

                    print(
                        "Please enter a valid number."
                    )
                    contribution_updates = None
                    break

                notes = input(
                    "Contribution notes: "
                ).strip()

                contribution_updates[student] = {
                    "hours": hours,
                    "notes": notes
                }

            if contribution_updates is None:
                continue

            result = certificate.complete_event(
                event_id,
                contribution_updates
            )

            print(result[1])

            # Generate certificates after successful completion
            if result[0]:

                print("\nGenerating certificates...")

                for student in accepted_students:

                    certificate_result = (
                        certificate.generate_certificate(
                            event_id,
                            student
                        )
                    )

                    print(certificate_result[1])


        elif choice == "4":

            print("Logging out...")
            break



def volunteer_menu(username, user_record):

    while True:

        print("\n===== VOLUNTEER MENU =====")
        print("1. View Eligible Events")
        print("2. Apply to an Event")
        print("3. View My Certificates")
        print("4. Logout")

        choice = get_valid_choice(
            "Enter choice: ",
            ["1", "2", "3", "4"]
        )

        
        if choice == "1":

            eligible_events = events.find_eligible_events(
                user_record.get("skills", {})
            )

            print("\n===== ELIGIBLE EVENTS =====")

            if not eligible_events:

                print("No eligible events found.")

            else:

                for event_id, event in eligible_events:

                    print("\nEvent ID:", event_id)
                    print(
                        "Name:",
                        event.get("name", "")
                    )
                    print(
                        "Description:",
                        event.get("description", "")
                    )
                    print(
                        "Organizer:",
                        event.get("organizer", "")
                    )
                    print(
                        "Required Skills:",
                        event.get("required_skills", {})
                    )
                    print(
                        "Slots:",
                        event.get("slots_needed", 0)
                    )
                    print(
                        "Status:",
                        event.get("status", "")
                    )


        elif choice == "2":

            print("\n===== APPLY TO AN EVENT =====")

            eligible_events = events.find_eligible_events(
                user_record.get("skills", {})
            )

            if not eligible_events:

                print("No eligible events found.")
                continue

            print("\nAvailable events:")

            for event_id, event in eligible_events:

                print(
                    event_id,
                    "-",
                    event.get("name", "Unnamed Event")
                )

            event_id = input(
                "\nEnter event ID to apply: "
            ).strip()

            # Make sure the selected event is in the eligible-event list
            eligible_ids = [
                event_id
                for event_id, event in eligible_events
            ]

            if event_id not in eligible_ids:

                print(
                    "Invalid event ID or you are not eligible "
                    "for this event."
                )
                continue

            result = events.apply_to_event(
                event_id,
                username
            )

            print(result[1])


        elif choice == "3":

            print("\n===== MY CERTIFICATES =====")

            certificate_folder = "certificates"

            if not os.path.exists(certificate_folder):

                print("No certificates found.")

            else:

                found = False

                for filename in os.listdir(
                    certificate_folder
                ):

                    if not filename.endswith(".txt"):
                        continue

                    # Certificate filenames are: event_id_username.txt
                    # check whether the filename ends with _username.txt
                    
                    if filename.endswith(
                        "_" + username + ".txt"
                    ):

                        print(filename)
                        found = True

                if not found:
                    print("No certificates found.")


        elif choice == "4":

            print("Logging out...")
            break



def main():

    while True:

        print("\n===== SKILLMATCH =====")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        choice = get_valid_choice(
            "Enter choice: ",
            ["1", "2", "3"]
        )


        if choice == "1":

            print("\n===== REGISTER =====")

            username = input(
                "Enter username: "
            ).strip()

            password = input(
                "Enter password: "
            )

            print("\nChoose role:")
            print("1. Organizer")
            print("2. Student Volunteer")

            role_choice = get_valid_choice(
                "Enter choice: ",
                ["1", "2"]
            )

            name = input(
                "Enter full name: "
            ).strip()

            if role_choice == "1":

                result = auth.register_user(
                    username,
                    password,
                    "Organizer",
                    name
                )

            else:

                dept = input(
                    "Enter department: "
                ).strip()

                skills = {}

                print(
                    "\nEnter your skills."
                )
                print(
                    "Example: Python:4, CAD:3"
                )
                print(
                    "Enter 'none' if you have no skills."
                )

                skills_input = input(
                    "Skills: "
                ).strip()

                if skills_input.lower() != "none" and skills_input != "":

                    try:

                        skill_list = skills_input.split(",")

                        for skill_item in skill_list:

                            skill, rating = (
                                skill_item.split(":")
                            )

                            skill = skill.strip()
                            rating = int(
                                rating.strip()
                            )

                            if rating < 0:
                                raise ValueError

                            skills[skill] = rating

                    except ValueError:

                        print(
                            "Invalid skill format."
                        )
                        print(
                            "Please use: "
                            "Python:4, CAD:3"
                        )
                        continue

                result = auth.register_user(
                    username,
                    password,
                    "Student Volunteer",
                    name,
                    dept=dept,
                    skills=skills
                )

            print(result[1])


        elif choice == "2":

            print("\n===== LOGIN =====")

            username = input(
                "Enter username: "
            ).strip()

            password = input(
                "Enter password: "
            )

            user_record = auth.login_user(
                username,
                password
            )

            if user_record is None:

                print(
                    "Invalid username or password."
                )

                continue

            print(
                "\nWelcome,",
                user_record.get("name", username)
            )

            # The username is NOT stored inside user_record.
            # using the username entered during login.

            role = user_record.get("role")

            if role == "Organizer":

                organizer_menu(username)

            elif role == "Student Volunteer":

                volunteer_menu(
                    username,
                    user_record
                )

            else:

                print(
                    "Unknown user role."
                )

        elif choice == "3":

            print("\nExiting SkillMatch...")
            break


if __name__ == "__main__":
    main()



===== SKILLMATCH =====
1. Register
2. Login
3. Exit


Enter choice:  1



===== REGISTER =====


Enter username:  nv
Enter password:  sss



Choose role:
1. Organizer
2. Student Volunteer


Enter choice:  1
Enter full name:  kk


Registered successfully

===== SKILLMATCH =====
1. Register
2. Login
3. Exit


Enter choice:  3



Exiting SkillMatch...


In [12]:
#data_store
"""
data_store.py — Member 1's responsibility (shared utility used by everyone)

Every other module reads/writes JSON ONLY through these two functions.
No other file should call open()/json.load()/json.dump() directly except
certificate.py, which writes plain .txt letters (not JSON).
"""

import json
import os

DATA_DIR = "data"


def load(filename):
    """
    Load and return a dict from a JSON file inside DATA_DIR.
    Returns an empty dict {} if the file doesn't exist yet
    (so the very first run of the program doesn't crash).
    """
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        return {}
    with open(path, "r") as f:
        return json.load(f)


def save(filename, data):
    """
    Save a dict to a JSON file inside DATA_DIR.
    Creates the DATA_DIR folder automatically if it doesn't exist.
    """
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, filename)
    with open(path, "w") as f:
        json.dump(data, f, indent=4)


In [13]:
#certificate
"""
certificate.py — Member 4's responsibility
Core Role: event completion workflow + generating .txt Completion Letters.

No JSON here for the letters themselves — these are plain text files
written with open()/.write(), per the "pure Python" constraint.
"""

import datetime
import os
import data_store

EVENTS_FILE = "events.json"
APPLICATIONS_FILE = "applications.json"
USERS_FILE = "users.json"
CERT_FOLDER = "certificates"


def complete_event(event_id, contribution_updates):
    """
    Mark an event's status as "completed" and record final hours/notes
    for every accepted student.

    contribution_updates: dict like
        {"aditi101": {"hours": 6, "notes": "Handled mic setup all evening"}}

    TODO:
      1. events = data_store.load(EVENTS_FILE); set events[event_id]["status"] = "completed"; save
      2. applications = data_store.load(APPLICATIONS_FILE)
      3. for username, update in contribution_updates.items():
             applications[event_id][username]["hours"] = update["hours"]
             applications[event_id][username]["notes"] = update["notes"]
      4. data_store.save(APPLICATIONS_FILE, applications)
    """
    pass


def generate_certificate(event_id, username):
    """
    Build and save a formatted .txt completion letter for one student,
    for one event, into CERT_FOLDER/{event_id}_{username}.txt

    Must include: student name, department, event name, date, role played,
    hours contributed, and an organizer sign-off line.

    TODO:
      1. events = data_store.load(EVENTS_FILE); event = events[event_id]
      2. users = data_store.load(USERS_FILE); student = users[username]
      3. applications = data_store.load(APPLICATIONS_FILE); app = applications[event_id][username]
      4. Only proceed if event["status"] == "completed" and app["status"] == "accepted"
         (this enforces the "strict policy" that you can't get a certificate
         for an event that isn't finished, or a role you weren't accepted for)
      5. os.makedirs(CERT_FOLDER, exist_ok=True)
      6. build a formatted multi-line string, something like:

         ==========================================
                 CERTIFICATE OF COMPLETION
         ==========================================
         This certifies that {student["name"]} ({student["dept"]})
         volunteered for "{event["name"]}" on {today's date}.

         Role: {app["role"]}
         Hours Contributed: {app["hours"]}
         Notes: {app["notes"]}

         Organizer: {event["organizer"]}
         ==========================================

      7. path = os.path.join(CERT_FOLDER, f"{event_id}_{username}.txt")
      8. with open(path, "w") as f: f.write(letter_text)
    """
    pass


In [14]:
#events
"""
events.py — Member 2's responsibility
Core Role: event creation, the matching search engine, and application state.

Data shapes (agree on these before writing logic):

events = {
    "E01": {
        "name": "Tech Fest Setup",
        "description": "Stage sound and lighting setup",
        "organizer": "priya_organizer",
        "required_skills": {"Sound Systems": 2, "Electrical": 1},
        "slots_needed": 3,
        "status": "open"              # "open" -> "in_progress" -> "completed"
    }
}

applications = {
    "E01": {
        "aditi101": {"status": "pending", "role": None, "hours": 0, "notes": ""}
    }
}
"""

import data_store
import auth

EVENTS_FILE = "events.json"
APPLICATIONS_FILE = "applications.json"


def create_event(event_id, name, description, organizer_username, required_skills, slots_needed):
    """
    Create a new event and save it. Starting status is always "open".

    TODO:
      1. events = data_store.load(EVENTS_FILE)
      2. check event_id not already used
      3. build the event dict matching the shape above
      4. events[event_id] = record
      5. data_store.save(EVENTS_FILE, events)
    """
    pass


def find_eligible_events(student_skills):
    """
    Loop through every event with status == "open", and use
    auth.check_eligibility() to keep only the ones this student qualifies for.

    Returns a list of (event_id, event_dict) tuples for display in the menu.

    TODO:
      events = data_store.load(EVENTS_FILE)
      results = []
      for event_id, event in events.items():
          if event["status"] == "open" and auth.check_eligibility(student_skills, event["required_skills"]):
              results.append((event_id, event))
      return results
    """
    pass


def apply_to_event(event_id, username):
    """
    Record a "pending" application for this student under this event.

    Strict policy checks to include (return (False, reason) if any fail):
      - event must exist and be "open"
      - student must not have already applied to this event
      - number of "accepted" applicants must be < event's slots_needed

    TODO:
      1. load applications, load events
      2. run the checks above
      3. applications.setdefault(event_id, {})[username] = {"status": "pending", "role": None, "hours": 0, "notes": ""}
      4. save applications
      5. return (True, "Applied successfully")
    """
    pass


def update_application_status(event_id, username, new_status, role=None):
    """
    Organizer accepts or rejects a pending applicant.
    new_status should be "accepted" or "rejected".
    If accepted, also store the role they'll play at the event.

    TODO: load applications, update the entry, save
    """
    pass


In [8]:
#seed_data
import auth
import events
def run():
    auth.register_user("priya_organizer", "pass123", "Organizer", "Priya Nair")
    auth.register_user("aditi101", "pass123", "Student Volunteer", "Aditi Sharma",
    dept="CSE", skills={"Python": 4, "Sound Systems": 3})
    auth.register_user("rohan102", "pass123", "Student Volunteer", "Rohan Mehta",
    dept="Mech", skills={"CAD": 4})
    events.create_event("E01", "Tech Fest Setup", "Stage sound and lighting setup",
    "priya_organizer", {"Sound Systems": 2}, slots_needed=2)
    print("Seed data loaded. Run main.py and log in with the accounts above.")
if __name__ == "__main__":
    run()

Seed data loaded. Run main.py and log in with the accounts above.


In [17]:
#auth
import data_store
USERS_FILE = "users.json"
def register_user(username, password, role, name, dept=None, skills=None):
    users = data_store.load(USERS_FILE)
    if username in users:
        return (False, "Username already exists")
    users[username] = {
        "password": password,
        "role": role,
        "name": name,
        "dept": dept,
        "skills": skills if skills else {}
    }
    data_store.save(USERS_FILE, users)
    return (True, "Registered successfully")
def login_user(username, password):
    users = data_store.load(USERS_FILE)
    user = users.get(username)
    if user and user["password"] == password:
        return user
    return None
def check_eligibility(student_skills, required_skills):
    for skill, min_rating in required_skills.items():
        if skill not in student_skills or student_skills[skill] < min_rating:
            return False
    return True
if __name__ == "__main__":
    result = register_user("aditi101", "pass123", "Student Volunteer", "Aditi Sharma", dept="CSE", skills={"Python": 4})
    print("Register result:", result)
    login_result = login_user("aditi101", "pass123")
    print("Login result:", login_result)
    wrong_login = login_user("aditi101", "wrongpass")
    print("Wrong password result:", wrong_login)
    check1 = check_eligibility({"Python": 4}, {"Python": 3})
    print("Eligibility check 1 (should be True):", check1)
    check2 = check_eligibility({"CAD": 4}, {"Python": 3})
    print("Eligibility check 2 (should be False):", check2)
    check3 = check_eligibility({"Python": 2}, {"Python": 3})
    print("Eligibility check 3 (should be False):", check3)

Register result: (False, 'Username already exists')
Login result: {'password': 'pass123', 'role': 'Student Volunteer', 'name': 'Aditi Sharma', 'dept': 'CSE', 'skills': {'Python': 4, 'Sound Systems': 3}}
Wrong password result: None
Eligibility check 1 (should be True): True
Eligibility check 2 (should be False): False
Eligibility check 3 (should be False): False
